# Détection de fraude — cycle de vie d'un modèle avec MLflow

Ce notebook accompagne la démo. Il déroule pas à pas ce que MLflow apporte :

1. **Tracking** — entraîner deux modèles et comparer leurs *runs*
2. **Model Registry** — versionner et aliaser le modèle servi en prod
3. **Promotion** — déplacer l'alias `champion` sans toucher au code applicatif

> Prérequis : un serveur MLflow doit tourner (voir README). On suppose ici
> `MLFLOW_TRACKING_URI=http://127.0.0.1:5000`.

In [ ]:
import mlflow, mlflow.sklearn
import pandas as pd
from mlflow.tracking import MlflowClient
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

FEATURES = ['amount','hour','distance_from_home_km','is_new_device',
            'is_foreign_country','transactions_last_24h','avg_amount_last_30d','amount_to_avg_ratio']
TARGET = 'is_fraud'
MODEL_NAME = 'FraudDetector'

mlflow.set_tracking_uri('http://127.0.0.1:5000')
mlflow.set_experiment('fraud_detection')

## 1. Les données

Transactions bancaires synthétiques. La fraude est **rare (~1,2 %)** : c'est tout l'enjeu.

In [ ]:
df = pd.read_csv('../fraud_data/transactions.csv')
print(f"{len(df)} transactions, taux de fraude = {df[TARGET].mean():.2%}")
X, y = df[FEATURES], df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
df.head()

## 2. Tracking — deux runs comparables

On entraîne deux modèles dans deux *runs* MLflow distincts. Chaque run
enregistre ses hyperparamètres, ses métriques et le modèle.

**Pourquoi pas seulement l'accuracy ?** Sur 1,2 % de fraude, un modèle qui
répond toujours « légitime » obtient ~99 % d'accuracy en ratant 100 % des
fraudes. On suit donc surtout le **recall** (part des fraudes détectées).

In [ ]:
def train_and_log(run_name, algo_name, estimator, register=False):
    with mlflow.start_run(run_name=run_name):
        pipe = Pipeline([('scaler', StandardScaler()), ('classifier', estimator)])
        pipe.fit(X_train, y_train)
        pred = pipe.predict(X_test)
        metrics = {
            'accuracy': accuracy_score(y_test, pred),
            'precision': precision_score(y_test, pred, zero_division=0),
            'recall': recall_score(y_test, pred, zero_division=0),
            'f1_score': f1_score(y_test, pred, zero_division=0),
        }
        mlflow.log_param('algorithm', algo_name)
        mlflow.log_metrics(metrics)
        kwargs = dict(registered_model_name=MODEL_NAME) if register else {}
        mlflow.sklearn.log_model(pipe, name='model', input_example=X_test.head(2), **kwargs)
        print(f"{run_name:18s} -> recall={metrics['recall']:.3f}  f1={metrics['f1_score']:.3f}  acc={metrics['accuracy']:.4f}")
        return metrics

In [ ]:
# v1 : le mauvais modele (arbre peu profond, sans reequilibrage)
train_and_log('v1_decision_tree', 'DecisionTreeClassifier',
              DecisionTreeClassifier(max_depth=2, random_state=42), register=True)

# v2 : le bon modele (foret aleatoire reequilibree)
train_and_log('v2_random_forest', 'RandomForestClassifier',
              RandomForestClassifier(n_estimators=300, class_weight='balanced', n_jobs=-1, random_state=42), register=True)

👉 **Ouvre maintenant l'UI MLflow** (`http://127.0.0.1:5000`) et compare les
deux runs côte à côte. L'accuracy de v1 est excellente (~99 %) alors que son
recall est nul : il ne détecte **aucune** fraude. C'est exactement le piège
que MLflow rend visible.

## 3. Model Registry — versionner et aliaser

Les deux `log_model(..., registered_model_name='FraudDetector')` ont créé les
versions 1 et 2. On taggue chaque version et on pose l'alias `champion` sur le
**mauvais** modèle (v1) pour reproduire la situation de départ en prod.

In [ ]:
client = MlflowClient()
versions = client.search_model_versions(f"name='{MODEL_NAME}'")
for v in sorted(versions, key=lambda v: int(v.version)):
    print(f"version {v.version} | run {v.run_id[:8]}")

client.set_model_version_tag(MODEL_NAME, '1', 'algo', 'DecisionTreeClassifier')
client.set_model_version_tag(MODEL_NAME, '2', 'algo', 'RandomForestClassifier')
client.set_registered_model_alias(MODEL_NAME, 'champion', '1')
print("\nAlias 'champion' -> version 1 (le mauvais modele, comme en prod)")

> 💡 Les *stages* `Staging`/`Production` sont **dépréciés depuis MLflow 2.9**.
> On utilise désormais des **alias** (`@champion`) et des **tags**. L'URI de
> service devient `models:/FraudDetector@champion`.

## 4. Servir le champion (ce que fait l'app)

L'application charge toujours `models:/FraudDetector@champion` — jamais un
numéro de version en dur. Testons les deux scénarios de la démo.

In [ ]:
normal = pd.DataFrame([[50.0,14,8.0,0,0,3,60.0,0.83]], columns=FEATURES)
fraude = pd.DataFrame([[4200.0,3,1400.0,1,1,7,150.0,28.0]], columns=FEATURES)

def serve_and_score():
    model = mlflow.sklearn.load_model(f'models:/{MODEL_NAME}@champion')
    mv = client.get_model_version_by_alias(MODEL_NAME, 'champion')
    print(f"Champion = v{mv.version} ({mv.tags.get('algo','?')})")
    for name, row in [('normal', normal), ('fraude', fraude)]:
        p = model.predict_proba(row)[0][1]
        verdict = 'FRAUDE' if p >= 0.5 else 'legitime'
        print(f"  {name:8s} score={p:.3f} -> {verdict}")

serve_and_score()

La fraude évidente est notée **sous le seuil** : le modèle en prod la laisse
passer. C'est le moment de silence de la démo. 🤐

## 5. Promotion en live

On déplace l'alias `champion` vers la v2. **Aucun code applicatif ne change** :
l'app rechargera simplement le nouveau champion au prochain rafraîchissement.

In [ ]:
client.set_registered_model_alias(MODEL_NAME, 'champion', '2')
print("Alias 'champion' -> version 2 (RandomForest)\n")
serve_and_score()

Même transaction, **verdict inversé** : la fraude est maintenant détectée.

> En une ligne (`set_registered_model_alias`), on a fait passer un nouveau
> modèle en production, de façon tracée, réversible et auditable. C'est ça,
> le cycle de vie MLflow.